In [1]:
import tensorflow as tf
print(tf)

<module 'tensorflow' from '/usr/local/lib/python3.11/dist-packages/tensorflow/__init__.py'>


In [3]:
from tensorflow.keras import layers, models
import tensorflow as tf

# Define the model architecture
def build_model(input_shape):
    model = models.Sequential()
    model.add(layers.Input(shape=input_shape))
    # First convolutional hidden layer
    model.add(layers.Conv2D(64, (3, 3), strides=2, padding='same', activation='tanh'))
    model.add(layers.BatchNormalization())

    # Second hidden layer (GRU)
    # Reshape for GRU input: flatten spatial dimensions into one dimension
    shape_before_gru = model.output_shape
    flattened_size = shape_before_gru[1] * shape_before_gru[2] * shape_before_gru[3]
    model.add(layers.Reshape((-1, shape_before_gru[-1])))
    model.add(layers.GRU(256, return_sequences=True))


    # Third hidden layer (transpose convolution)
    # Reshape back to image dimensions before transpose convolution
    # Calculate the expected shape after GRU and before the second reshape
    shape_after_gru = model.output_shape
    # Calculate the target shape for Conv2DTranspose based on the original input shape
    # The spatial dimensions are doubled due to the stride of 2 in Conv2DTranspose
    target_h = tf.shape(model.input)[1]
    target_w = tf.shape(model.input)[2]

    # Reshape to a shape compatible with Conv2DTranspose
    # The number of elements should match the output of the GRU layer
    # We use -1 for the spatial dimensions and the last dimension for the features from GRU output
    model.add(layers.Reshape((target_h, target_w, shape_after_gru[-1])))


    model.add(layers.Conv2DTranspose(64, (3, 3), strides=2, padding='same', activation='tanh'))
    model.add(layers.BatchNormalization())

    # Final output layer
    model.add(layers.Conv2D(1, (3, 3), padding='same', activation='sigmoid'))

    return model

In [4]:
from skimage import io
image = io.imread('/content/demo.jpg')

In [6]:
model = build_model(image.shape)

TypeError: unsupported operand type(s) for *: 'int' and 'NoneType'